# Retrieval Judging Prompt Builder

Purpose:
    This notebook runs the retrieval-only part of the RAG workflow and writes
    one Markdown grading prompt per question for manual external LLM judging.

Flow:
1. Load `test_document.docx`.
2. Parse it with `preprocessing.assessment_processor.parse_to_dict(...)`.
3. Build the draft retrieval store with the inference helpers.
4. Run RAG retrieval with no external answer-generation LLM.
5. Insert the retrieval payload into `prompt_skeleton.md`.
6. Write one question prompt into `external_llm_evaluation/` for each test question.

The notebook evaluates retrieval evidence only. It does not score final model
answers and it does not save intermediate JSON payload files.


In [1]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import JSON, Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
    if (candidate / "llm_rag").exists() and (candidate / "preprocessing").exists()
)
RETRIEVAL_JUDGING_DIR = REPO_ROOT / "llm_rag" / "evaluation" / "retrieval_judging"
RAW_REFERENCE_DIR = REPO_ROOT / "llm_rag" / "i_raw_documents"
PROMPT_OUTPUT_DIR = RETRIEVAL_JUDGING_DIR / "external_llm_evaluation"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.assessment_processor import parse_to_dict
from llm_rag.iv_inference.rag_runtime import answer_rag_question
from llm_rag.iv_inference.rag_runtime import build_rag_debug_payload
from llm_rag.iv_inference.rag_runtime import build_report_from_assessment
from llm_rag.iv_inference.rag_runtime import ensure_draft_store_from_report

print(f"Repo root: {REPO_ROOT}")
print(f"Retrieval judging dir: {RETRIEVAL_JUDGING_DIR}")


c:\Users\utsav\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo root: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer
Retrieval judging dir: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\llm_rag\evaluation\retrieval_judging


In [2]:
DOCX_PATH = RETRIEVAL_JUDGING_DIR / "test_document.docx"
PROMPT_SKELETON_PATH = RETRIEVAL_JUDGING_DIR / "prompt_skeleton.md"

QUESTIONS = [
    "Based on Extent of occurrence (EOO) and Area of occupancy (AOO) only, what IUCN category would the uploaded assessment support?",
    "What evidence does the uploaded assessment provide for number of locations and severe fragmentation under Criterion B?",
    "What supporting information is required or recommended for a threatened Red List assessment?",
    "What mapping or spatial-data checks should be considered when reviewing this assessment?",
    "Does the uploaded assessment include the information needed to justify population trend and continuing decline?",
    "What should be checked in the assessment rationale for consistency with IUCN Red List guidance?",
]

reference_documents = sorted(path.name for path in RAW_REFERENCE_DIR.glob("*.pdf"))
prompt_skeleton = PROMPT_SKELETON_PATH.read_text(encoding="utf-8")

display(Markdown("## Evaluation Inputs"))
display(JSON({
    "docx_path": str(DOCX_PATH.relative_to(REPO_ROOT)),
    "prompt_skeleton": str(PROMPT_SKELETON_PATH.relative_to(REPO_ROOT)),
    "prompt_output_dir": str(PROMPT_OUTPUT_DIR.relative_to(REPO_ROOT)),
    "question_count": len(QUESTIONS),
    "reference_documents": reference_documents,
}))


## Evaluation Inputs

<IPython.core.display.JSON object>

In [3]:
assessment = parse_to_dict(str(DOCX_PATH))
report_dict = build_report_from_assessment(assessment)
report_signature = f"{DOCX_PATH.resolve()}::{len(report_dict)}"
draft_store = ensure_draft_store_from_report({}, report_dict, report_signature)

display(Markdown("## Parsed Draft Summary"))
display(JSON({
    "assessment_title": assessment.get("title"),
    "top_level_children": len(assessment.get("children", [])),
    "report_section_count": len(report_dict),
    "draft_store_chunk_count": len(draft_store),
}))

pd.DataFrame({
    "section_path": list(report_dict.keys())[:20],
    "html_preview": [value[:180].replace("\n", " ") for value in list(report_dict.values())[:20]],
})


## Parsed Draft Summary

<IPython.core.display.JSON object>

,section_path,html_preview
0,test_document,<p><b>Draft</b></p> <p><b><i>Bulbostylis atrac...
1,test_document > Red List Assessment > Assessme...,"<p><b>Assessor(s): </b>Lemboye, B. & Lyu, E</p..."
2,test_document > Red List Assessment > Assessme...,<p>Bulbostylis atracuminata occurs in a restri...
3,test_document > Distribution > Geographic Range,<p><i>Bulbostylis atracuminata (Larridon et al...
4,test_document > Distribution > Area of Occupan...,<table> <tr><td><b>Estimated area of occupan...
5,test_document > Distribution > Locations Infor...,<table> <tr><td><b>Number of Locations</b></...
6,test_document > Distribution > Very restricted...,<table> <tr><td><b>Very restricted in area o...
7,test_document > Distribution > Elevation / Dep...,<p><b>Elevation Lower Limit (in metres above s...
8,test_document > Distribution > Map Status,<table> <tr><td><b>Map Status</b></td><td><b...
9,test_document > Distribution > Biogeographic R...,<p><b>Biogeographic Realm: </b>Afrotropical</p>


In [4]:
PROMPT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cases: list[dict[str, Any]] = []

for question_number, question in enumerate(QUESTIONS, start=1):
    response = answer_rag_question(
        query=question,
        draft_store=draft_store,
        llm_config=None,
    )
    debug_payload = build_rag_debug_payload(response)
    reference_payload = response.get("reference_payload", {})
    reference_results = reference_payload.get("results") or []

    retrieved_reference_items = []
    for rank, candidate in enumerate(reference_results, start=1):
        metadata = getattr(candidate, "metadata", {}) or {}
        item = {
            "rank": rank,
            "block_type": metadata.get("block_type"),
            "source": metadata.get("source_file"),
            "page": metadata.get("page"),
            "section": metadata.get("section_title") or metadata.get("table_title"),
            "text": getattr(candidate, "text", ""),
        }
        parent_text = getattr(candidate, "parent_text", None)
        if parent_text:
            item["parent_text"] = parent_text
        retrieved_reference_items.append(item)

    draft_hits = []
    for rank, hit in enumerate(response.get("draft_hits") or [], start=1):
        draft_hits.append({
            "rank": rank,
            "section_path": hit.get("section_path"),
            "source_key": hit.get("source_key"),
            "score": hit.get("score"),
            "text": hit.get("text"),
        })

    retrieval_payload = {
        "question": question,
        "route": response.get("route"),
        "deterministic_answer": response.get("deterministic_answer"),
        "reference_documents": reference_documents,
        "answer_scaffold": reference_payload.get("answer_scaffold"),
        "subqueries": reference_payload.get("subqueries"),
        "retrieved_reference_items": retrieved_reference_items,
        "draft_hits": draft_hits,
    }

    judge_prompt = prompt_skeleton.replace(
        "{...JSON loaded...}",
        json.dumps(retrieval_payload, ensure_ascii=False, indent=2),
    )
    safe_question = re.sub(r"[^a-zA-Z0-9]+", "_", question.lower()).strip("_")[:60]
    prompt_path = PROMPT_OUTPUT_DIR / f"question_{question_number:02d}_{safe_question}.md"
    prompt_path.write_text(judge_prompt, encoding="utf-8")

    cases.append({
        "question_number": question_number,
        "question": question,
        "response": response,
        "debug_payload": debug_payload,
        "retrieval_payload": retrieval_payload,
        "judge_prompt_path": prompt_path,
    })

summary = pd.DataFrame([
    {
        "question_number": case["question_number"],
        "question": case["question"],
        "route": case["response"].get("route"),
        "reference_item_count": len(case["retrieval_payload"].get("retrieved_reference_items", [])),
        "draft_hit_count": len(case["retrieval_payload"].get("draft_hits", [])),
        "judge_prompt": str(case["judge_prompt_path"].relative_to(REPO_ROOT)),
    }
    for case in cases
])

display(Markdown("## Generated Retrieval-Judging Prompts"))
summary


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 481.75it/s, Materializing param=pooler.dense.weight]                               


## Generated Retrieval-Judging Prompts

,question_number,question,route,reference_item_count,draft_hit_count,judge_prompt
0,1,Based on Extent of occurrence (EOO) and Area o...,hybrid_rag,8,6,llm_rag\evaluation\retrieval_judging\external_...
1,2,What evidence does the uploaded assessment pro...,deterministic_plus_hybrid_rag,8,6,llm_rag\evaluation\retrieval_judging\external_...
2,3,What supporting information is required or rec...,hybrid_rag,8,6,llm_rag\evaluation\retrieval_judging\external_...
3,4,What mapping or spatial-data checks should be ...,hybrid_rag,8,6,llm_rag\evaluation\retrieval_judging\external_...
4,5,Does the uploaded assessment include the infor...,hybrid_rag,8,6,llm_rag\evaluation\retrieval_judging\external_...
5,6,What should be checked in the assessment ratio...,hybrid_rag,8,6,llm_rag\evaluation\retrieval_judging\external_...


In [5]:
for case in cases:
    display(Markdown(f"# Question {case['question_number']}"))
    display(Markdown(f"**Question:** {case['question']}"))
    display(Markdown(f"**Prompt file:** `{case['judge_prompt_path'].relative_to(REPO_ROOT)}`"))

    display(Markdown("## Retrieval Payload"))
    display(JSON(case["retrieval_payload"]))

    display(Markdown("## Raw Debug Payload"))
    display(JSON(case["debug_payload"]))


# Question 1

**Question:** Based on Extent of occurrence (EOO) and Area of occupancy (AOO) only, what IUCN category would the uploaded assessment support?

**Prompt file:** `llm_rag\evaluation\retrieval_judging\external_llm_evaluation\question_01_based_on_extent_of_occurrence_eoo_and_area_of_occupancy_aoo_.md`

## Retrieval Payload

<IPython.core.display.JSON object>

...

**Prompt file:** `llm_rag\evaluation\retrieval_judging\external_llm_evaluation\question_06_what_should_be_checked_in_the_assessment_rationale_for_consi.md`

## Retrieval Payload

<IPython.core.display.JSON object>

## Raw Debug Payload

<IPython.core.display.JSON object>